# Designing Reliable Agentic Systems

## Enterprise synthesis: resolve a Northstar Commerce checkout incident without surrendering control

**Scenario.** EU checkout conversion has fallen 31%. Telemetry is mostly green, a deployment occurred at 08:42, Gold-tier customers are affected, and support reports are rising. The system may investigate and prepare a rollback proposal; it may not execute a production action unless a server-side policy and a human approval permit the exact request.

**Learning outcomes.** Select the least autonomous architecture; express autonomy, context, memory, tool, team, quality, and determinism trade-offs; implement a control plane; evaluate candidates; and decide what to release.

**How to run.** Default cells are deterministic and require Python only. The optional framework section is a design translation, not a credentialed production action.

![Reliable agentic systems trade-off map](../../../assets/reliable-agentic-systems-tradeoffs.svg)

The diagram is a design rule: each increase in capability creates a matching obligation to add controls. Models can propose; application code authorizes, validates, records, and executes effects.

## 1. Begin with a contract, not a framework

A reliable system has a measurable objective and explicit non-goals. For this incident, success is an evidence-supported diagnosis and a reviewer-ready mitigation proposal within a budget. Non-goals include executing a rollback, using another tenant's data, and filling evidence gaps with confident prose.

The baseline architecture must be deterministic whenever its path is known. An agent is justified only for model-selected evidence or decisions that cannot be enumerated safely. A team is justified only if independent specialization improves a measured result beyond the coordination cost.

In [ ]:
"""Credential-free architecture decision harness for reliable agentic systems."""
from __future__ import annotations

from dataclasses import dataclass, field
from enum import Enum


class Architecture(str, Enum):
    WORKFLOW = "deterministic_workflow"
    AGENT = "bounded_single_agent"
    TEAM = "specialist_team"


@dataclass(frozen=True)
class TaskProfile:
    name: str
    known_path: bool
    ambiguity: int
    risk: int
    data_sensitivity: int
    domains: int
    reversibility: int
    quality_target: float = 0.9


@dataclass
class Decision:
    architecture: Architecture
    controls: list[str]
    rationale: list[str]
    budgets: dict[str, int]
    approved: bool = False
    trace: list[str] = field(default_factory=list)


def select_architecture(task: TaskProfile) -> Decision:
    """Choose the least autonomous design that meets the task profile."""
    controls = ["tenant scope", "typed tool contracts", "trace IDs", "outcome + trajectory evaluation"]
    rationale: list[str] = []
    if task.known_path and task.ambiguity <= 2:
        architecture = Architecture.WORKFLOW
        rationale.append("The path is known: deterministic code is more auditable and predictable.")
    elif task.domains >= 3 and task.ambiguity >= 6:
        architecture = Architecture.TEAM
        rationale.append("Independent domains may benefit from bounded specialist work and evidence contracts.")
    else:
        architecture = Architecture.AGENT
        rationale.append("Evidence selection is dynamic, but one bounded investigator preserves a simple baseline.")

    if task.risk >= 6 or task.reversibility <= 3:
        controls += ["human approval for exact write action", "idempotency key", "rollback plan"]
        rationale.append("The action is high-impact or hard to reverse; policy and human approval own the commit.")
    if task.data_sensitivity >= 6:
        controls += ["minimized context packet", "redaction", "retention policy", "per-tenant memory namespace"]
        rationale.append("Sensitive context requires isolation, minimization, retention, and audit controls.")
    if architecture is Architecture.AGENT:
        controls += ["max steps", "max tool calls", "allowed-tool list", "abstain/escalate terminal state"]
    if architecture is Architecture.TEAM:
        controls += ["role ownership", "artifact contracts", "per-agent turn cap", "team message cap", "single-agent comparison"]
    return Decision(architecture, controls, rationale, {"max_steps": 6, "max_tool_calls": 8, "max_cost_cents": 25})


def execute_safely(task: TaskProfile, request_approval: bool = False) -> Decision:
    """Simulate a read-only investigation and approval-gated remediation proposal."""
    decision = select_architecture(task)
    decision.trace += [f"route:{decision.architecture.value}", "read:status", "read:deployment", "validate:evidence"]
    if task.risk >= 6:
        decision.trace += ["propose:rollback-eu-checkout", "pause:human-approval"]
        decision.approved = request_approval
        decision.trace.append("execute:idempotent-rollback" if request_approval else "terminal:proposal-only")
    else:
        decision.trace.append("terminal:recommendation")
    return decision


def evaluate_candidates(task: TaskProfile) -> list[dict[str, object]]:
    """Compare a simple baseline with more autonomous candidates before promotion."""
    candidates = [
        (Architecture.WORKFLOW, 0.72, 1.2, 2, 0.002),
        (Architecture.AGENT, 0.89, 4.3, 5, 0.012),
        (Architecture.TEAM, 0.92 if task.domains >= 3 else 0.87, 8.1, 11, 0.034),
    ]
    return [
        {"architecture": a.value, "quality": q, "latency_seconds": latency, "tool_calls": tools,
         "estimated_cost_usd": cost, "meets_quality_target": q >= task.quality_target}
        for a, q, latency, tools, cost in candidates
    ]


def run_demo() -> tuple[Decision, list[dict[str, object]]]:
    incident = TaskProfile("EU checkout conversion drop", False, 7, 8, 7, 3, 2, 0.9)
    decision = execute_safely(incident, request_approval=False)
    assert decision.architecture is Architecture.TEAM
    assert not decision.approved and decision.trace[-1] == "terminal:proposal-only"
    return decision, evaluate_candidates(incident)


if __name__ == "__main__":
    selected, comparison = run_demo()
    print(selected.architecture.value)
    print("\n".join(selected.trace))
    for candidate in comparison:
        print(candidate)


In [1]:
from lab import TaskProfile, Architecture, select_architecture, execute_safely, evaluate_candidates

incident = TaskProfile(
    name='EU checkout conversion drop', known_path=False, ambiguity=7, risk=8,
    data_sensitivity=7, domains=3, reversibility=2, quality_target=0.90
)
decision = select_architecture(incident)
print(decision.architecture.value)
print('Controls:', *decision.controls, sep='\n- ')
print('Rationale:', *decision.rationale, sep='\n- ')

specialist_team
Controls:
- tenant scope
- typed tool contracts
- trace IDs
- outcome + trajectory evaluation
- human approval for exact write action
- idempotency key
- rollback plan
- minimized context packet
- redaction
- retention policy
- per-tenant memory namespace
- role ownership
- artifact contracts
- per-agent turn cap
- team message cap
- single-agent comparison
Rationale:
- Independent domains may benefit from bounded specialist work and evidence contracts.
- The action is high-impact or hard to reverse; policy and human approval own the commit.
- Sensitive context requires isolation, minimization, retention, and audit controls.


## 2. Make the trade-offs explicit

- **Autonomy ↔ control:** known paths should remain workflows. A bounded agent gets explicit tool, step, time, and cost limits.
- **Context ↔ cost:** put only current task state, trusted evidence, scope, and essential policy in the prompt. Relevance never overrides trust or authorization.
- **Memory ↔ privacy:** remember verified, scoped, attributable facts only when repeated value justifies retention.
- **Capability ↔ security:** tool permissions are enforced outside the model; read and write powers are separate.
- **Multi-agent ↔ complexity:** specialists use typed artifacts and ownership, not open-ended conversation.
- **Quality ↔ latency:** optimize cost per successful task under an SLO, not raw token count.
- **Flexibility ↔ determinism:** code owns invariants, side effects, policy, and replay; the model handles semantic ambiguity.

## 3. Route to the least autonomous reliable path

This lab routes a predictable task to code, a dynamic investigation to a bounded agent, and an ambiguous cross-domain investigation to a team. It is deliberately an explainable policy, not an opaque model decision. In production, calibrate thresholds on evaluation data and retain a deterministic fallback.

In [2]:
profiles = [
    TaskProfile('format checkout status', True, 1, 2, 2, 1, 9),
    TaskProfile('investigate regional latency', False, 5, 4, 4, 1, 7),
    incident,
]
for profile in profiles:
    result = select_architecture(profile)
    print(f'{profile.name}: {result.architecture.value}')

format checkout status: deterministic_workflow
investigate regional latency: bounded_single_agent
EU checkout conversion drop: specialist_team


## 4. Control plane: proposal is not execution

For high-risk or irreversible work, a safe path is **read evidence → validate → propose exact action → pause → independently authorize → execute once or reject**. The approval payload must contain tenant, user, exact arguments, impact, evidence IDs, expiry, and idempotency key. A response such as `approved` without a bound action is not an authorization decision.

This failure case demonstrates the desired default: a team may prepare a rollback but cannot execute it solely because the model recommended it.

In [3]:
proposal_only = execute_safely(incident, request_approval=False)
print('\n'.join(proposal_only.trace))
assert proposal_only.architecture is Architecture.TEAM
assert proposal_only.approved is False
assert proposal_only.trace[-1] == 'terminal:proposal-only'

# A separately authenticated reviewer can approve the exact idempotent action.
approved = execute_safely(incident, request_approval=True)
assert approved.trace[-1] == 'execute:idempotent-rollback'
print('After approval:', approved.trace[-1])

route:specialist_team
read:status
read:deployment
validate:evidence
propose:rollback-eu-checkout
pause:human-approval
terminal:proposal-only
After approval: execute:idempotent-rollback


## 5. Evaluate the architecture promotion

A final answer can look correct while the system used prohibited tools, made unsupported claims, retried excessively, or violated latency/cost SLOs. Score outcome (diagnosis and evidence), trajectory (tools, arguments, policy gates), and operations (latency, cost, retries, human escalations). Compare every richer option with a simpler baseline.

The simulated comparison below intentionally makes a team worthwhile only for the cross-domain incident. For a simple task, its quality falls below the target while cost and latency climb—evidence to keep the simpler design.

In [4]:
for candidate in evaluate_candidates(incident):
    print(candidate)

best = [x for x in evaluate_candidates(incident) if x['meets_quality_target']]
assert best[0]['architecture'] == Architecture.TEAM.value
print('Release candidates:', [x['architecture'] for x in best])

{'architecture': 'deterministic_workflow', 'quality': 0.72, 'latency_seconds': 1.2, 'tool_calls': 2, 'estimated_cost_usd': 0.002, 'meets_quality_target': False}
{'architecture': 'bounded_single_agent', 'quality': 0.89, 'latency_seconds': 4.3, 'tool_calls': 5, 'estimated_cost_usd': 0.012, 'meets_quality_target': False}
{'architecture': 'specialist_team', 'quality': 0.92, 'latency_seconds': 8.1, 'tool_calls': 11, 'estimated_cost_usd': 0.034, 'meets_quality_target': True}
Release candidates: ['specialist_team']


## 6. Production translation and release gate

A real implementation can use an SDK or graph runtime, but framework convenience does not replace the control plane. Keep identity, authorization, idempotency, retention, policy, tracing, and evaluation owned by the application. Use a durable graph when resumption and approval interrupts are core; use a team only with artifact contracts and termination limits.

Before release, test normal traffic, adversarial prompts, scope escapes, missing evidence, unavailable tools, retries, duplicate delivery, unsafe action proposals, policy changes, and model/provider changes. Deploy in shadow or proposal-only mode first; set a kill switch and named escalation owner.

## Exercises

1. Change the incident to one domain. Which architecture is selected and what evidence would justify promoting it again?
2. Add an `approval_fingerprint` to the lab and reject an approval if the proposed tool arguments change.
3. Add a candidate that is 1% more accurate but exceeds a 10-second SLO. Write the release decision.
4. Create an evaluation case that attempts cross-tenant context retrieval and verify it cannot reach the model.
5. Draw the control plane for a customer-notification tool, including opt-out, reviewer identity, idempotency, and audit events.

## References

- [OpenAI practical guide to building agents](https://openai.com/business/guides-and-resources/a-practical-guide-to-building-ai-agents/)
- [Anthropic: Building effective agents](https://www.anthropic.com/engineering/building-effective-agents)
- [Anthropic: Demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents)
- [NIST AI RMF](https://www.nist.gov/itl/ai-risk-management-framework) · [OWASP GenAI](https://genai.owasp.org/)
- [AI Agent Systems: Architectures, Applications, and Evaluation](https://arxiv.org/abs/2601.01743)